# Analyse GDELT : Détection d'anomalies

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

# Chargement des données
df = pd.read_csv("../data/clean/gdelt_benin_clean.csv", low_memory=False)
df["date"] = pd.to_datetime(df["SQLDATE"], format="%Y-%m-%d")
df.head()

,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,Actor1Religion1Code,Actor1Religion2Code,Actor1Type1Code,Actor1Type2Code,Actor1Type3Code,Actor2Code,Actor2Name,Actor2CountryCode,Actor2KnownGroupCode,Actor2EthnicCode,Actor2Religion1Code,Actor2Religion2Code,Actor2Type1Code,Actor2Type2Code,Actor2Type3Code,IsRootEvent,EventCode,EventBaseCode,EventRootCode,QuadClass,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,Actor1Geo_Type,Actor1Geo_FullName,Actor1Geo_CountryCode,Actor1Geo_ADM1Code,Actor1Geo_ADM2Code,Actor1Geo_Lat,Actor1Geo_Long,Actor1Geo_FeatureID,Actor2Geo_Type,Actor2Geo_FullName,Actor2Geo_CountryCode,Actor2Geo_ADM1Code,Actor2Geo_ADM2Code,Actor2Geo_Lat,Actor2Geo_Long,Actor2Geo_FeatureID,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL,year,month,week,date
0,1248558842,2025-06-09,202506,2025,2025.4356,BEN,COTONOU,BEN,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,1,70,70,7,2,7.0,20,1,10,3.267974,1,Benin,BN,BN,NaN,9.5000,2.25000,BN,0,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,1,Benin,BN,BN,NaN,9.5,2.25,BN,20250609020000,https://english.news.cn/20250609/833e6be5b34f4...,2025,6,24,2025-06-09
1,1248561281,2025-06-09,202506,2025,2025.4356,NGA,BENIN CITY,NGA,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,1,173,173,17,4,-5.0,3,1,3,-3.353659,1,Benin,BN,BN,NaN,9.5000,2.25000,BN,0,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,1,Benin,BN,BN,NaN,9.5,2.25,BN,20250609023000,https://punchng.com/ndlea-intercepts-drugs-dis...,2025,6,24,2025-06-09
2,1248560915,2025-06-09,202506,2025,2025.4356,BEN,BENIN,BEN,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,NGA,BENIN CITY,NGA,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,1,173,173,17,4,-5.0,2,1,2,-3.353659,1,Benin,BN,BN,NaN,9.5000,2.25000,BN,1,Benin,BN,BN,NaN,9.5000,2.25000,BN,1,Benin,BN,BN,NaN,9.5,2.25,BN,20250609023000,https://punchng.com/ndlea-intercepts-drugs-dis...,2025,6,24,2025-06-09
3,1248556983,2025-06-09,202506,2025,2025.4356,AFR,AFRICA,AFR,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,USAGOV,US OFFICIAL,USA,NaN,NaN,NaN,NaN,GOV,Unknown,Unknown,0,40,40,4,1,1.0,2,1,2,-1.637612,4,"Agadez, Agadez, Niger",NG,NG01,22553,16.9733,7.99111,-1077349,4,"Agadez, Agadez, Niger",NG,NG01,22553,16.9733,7.99111,-1077349,1,Benin,BN,BN,NaN,9.5,2.25,BN,20250609013000,https://www.eurasiareview.com/09062025-us-afri...,2025,6,24,2025-06-09
4,1248557466,2025-06-09,202506,2025,2025.4356,USAGOV,US OFFICIAL,USA,NaN,NaN,NaN,NaN,GOV,Unknown,Unknown,AFR,AFRICA,AFR,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,0,40,40,4,1,1.0,1,1,1,-1.637612,4,"Agadez, Agadez, Niger",NG,NG01,22553,16.9733,7.99111,-1077349,4,"Agadez, Agadez, Niger",NG,NG01,22553,16.9733,7.99111,-1077349,1,Benin,BN,BN,NaN,9.5,2.25,BN,20250609013000,https://www.eurasiareview.com/09062025-us-afri...,2025,6,24,2025-06-09


## Filtrage Initial
On supprime les lignes sans coordonnées et sans score Goldstein.

On ne garde que les événements racines avant l'entraînement du modèle.

In [8]:
print(f"Taille du dataset initial : {len(df)}")

df_clean = df.dropna(subset=["ActionGeo_Lat", "ActionGeo_Long", "GoldsteinScale"]).copy()

# Le filtre crucial qui manquait avant l'entraînement :
df_clean = df_clean[df_clean["IsRootEvent"] == 1]

print(f"Taille après conservation des Root Events uniquement : {len(df_clean)}")

Taille du dataset initial : 22778
Taille après conservation des Root Events uniquement : 14496


## 2. Feature Engineering
Création des indicateurs de violence et de sévérité pondérée.

In [9]:
# Liste des codes CAMEO hostiles
VIOLENT_CODES = {
    14, 141, 142, 143, 144, 145,  # Protestations
    15, 151, 152, 153, 154,       # Menaces de force
    16, 160, 161, 162, 163, 164,  # Démonstrations de force
    17, 170, 171, 172, 173, 174,  # Réductions de droits
    18, 180, 181, 182, 183,       # Assauts
    19, 190, 191, 192, 193, 194,  # Combats
    20, 200, 201, 202, 203, 204,  # Violence de masse
}

df_clean["is_violent"] = df_clean["EventCode"].isin(VIOLENT_CODES).astype(int)

# Score de sévérité : GoldsteinScale inversé (les événements négatifs deviennent positifs)
df_clean["severity_score"] = (df_clean["GoldsteinScale"] * -1).clip(lower=0)

# Pondération par le volume médiatique
df_clean["weighted_severity"] = df_clean["severity_score"] * np.log1p(df_clean["NumArticles"])

## 3. Entraînement de l'Isolation Forest
Le modèle va maintenant chercher des anomalies parmi les événements uniques,
et non plus parmi les répétitions médiatiques.

In [10]:
features_for_isolation_forest = [
    'GoldsteinScale', 
    'NumMentions', 
    'NumSources', 
    'NumArticles', 
    'AvgTone'
]

# Retrait des éventuels NaN sur les colonnes cibles
df_model = df_clean.dropna(subset=features_for_isolation_forest).copy()

scaler = StandardScaler()
data_scaled = scaler.fit_transform(df_model[features_for_isolation_forest])

# Entraînement (5% de contamination estimée)
model = IsolationForest(
    n_estimators=100, 
    contamination=0.05, 
    random_state=42
)

model.fit(data_scaled)

# Prédiction (-1 = anomalie, 1 = normal)
df_model['anomaly_score'] = model.decision_function(data_scaled)
df_model['is_anomaly'] = model.predict(data_scaled)

print(df_model['is_anomaly'].value_counts())

is_anomaly
 1    13771
-1      725
Name: count, dtype: int64


## 4. Extraction et Nettoyage des Résultats
On isole les anomalies qui sont aussi des événements violents,
puis on supprime les doublons géographiques/temporels et les URL redondantes.

In [13]:
colonnes_cles = ['SQLDATE', 'Actor1Name', 'Actor2Name', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long']

anomalies_violentes = df_model[
    (df_model['is_anomaly'] == -1) & 
    (df_model['is_violent'] == 1)
].sort_values(by='weighted_severity', ascending=False) \
 .drop_duplicates(subset=colonnes_cles, keep='first') \
 .drop_duplicates(subset=['SOURCEURL'])

# Affichage
colonnes_affichage = [
    'date', 'Actor1Name', 'Actor2Name', 'EventCode', 
    'GoldsteinScale', 'AvgTone', 'NumArticles', 'SOURCEURL'
]

print("\n--- Top 15 des événements anormaux violents ---")
display(anomalies_violentes[colonnes_affichage].head(15))


--- Top 15 des événements anormaux violents ---


,date,Actor1Name,Actor2Name,EventCode,GoldsteinScale,AvgTone,NumArticles,SOURCEURL
16614,2025-06-06,POLICE OFFICER,BENIN,190,-10.0,-8.431615,20,https://www.spacewar.com/afp/250605222330.i2el...
10941,2026-01-08,BENIN,MEDIA,190,-10.0,-4.477108,20,https://www.zawya.com/en/press-release/africa-...
20236,2025-08-19,TOGO,Unknown,190,-10.0,-7.544113,18,https://www.yahoo.com/news/articles/togo-tight...
22526,2025-08-27,BENIN,Unknown,180,-9.0,-2.882401,20,https://maliactu.net/mopti-une-initiative-cito...
17771,2025-11-04,Unknown,POLICE,181,-9.0,-6.332498,14,https://leadership.ng/police-arrest-woman-for-...
2253,2025-04-27,TERRORIST,BENIN,190,-10.0,-7.983193,10,https://fr.allafrica.com/stories/202504270077....
20524,2025-09-24,ROBBER,BENIN,193,-10.0,-11.267606,10,https://promptnewsonline.com/jewellery-thirsty...
2227,2025-06-06,POLICE,BENIN CITY,190,-10.0,-13.452915,10,https://dailypost.ng/2025/06/06/police-arrest-...
983,2025-06-08,BENIN,POLICE OFFICER,190,-10.0,-7.547170,10,https://www.terradaily.com/reports/Suspected_j...
1735,2026-04-06,VILLAGE,Unknown,190,-10.0,-9.467456,10,https://www.aftenposten.no/verden/i/k00lgA/min...
